In [7]:
import pandas as pd
import json
import numpy as np
from glob import glob
from tqdm import tqdm
import os
import sys
import pdb

rdir = '../results_tuned/'
params_dir = './methods/'
figdir = f'../postprocessing/figs/black-box-tuned/'
overwrite = True

print('reading results from  directory', rdir)

reading results from  directory ../results_tuned/


In [2]:
frames = []
fails = []
for f in tqdm(glob(rdir + '/*/*.json')): # _cv_results
    try: 
        r = json.load(open(f,'r'))
        sub_r = {k:v for k,v in r.items()}
        frames.append(sub_r) 
    except Exception as e:
        fails.append([f,e])
        pass

  0%|          | 0/966 [00:00<?, ?it/s]

100%|██████████| 966/966 [00:01<00:00, 527.83it/s]


In [3]:
df_results = pd.DataFrame.from_records(frames)
df_results['params_str'] = df_results['params'].apply(str)
print(df_results.shape)
df_results

(966, 12)


,dataset,algorithm,params,random_state,time_time,mse_train,mae_train,r2_train,mse_test,mae_test,r2_test,params_str
0,1029_LEV,tunedafp,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*...",15795,1135.189426,0.484815,0.542864,0.458762,0.596256,0.607939,0.375737,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*..."
1,1029_LEV,tunedafp,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*...",23654,1190.109223,0.578843,0.599923,0.366599,0.598478,0.594979,0.333769,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*..."
2,1029_LEV,tunedafp,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*...",860,1150.261674,0.608407,0.618654,0.304413,0.637304,0.652530,0.372979,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*..."
3,1029_LEV,tunedafp_ehc,"{'g': 100, 'op_list': ['n', 'v', '+', '-', '*'...",15795,646.173966,0.429641,0.517089,0.520358,0.550436,0.581564,0.423710,"{'g': 100, 'op_list': ['n', 'v', '+', '-', '*'..."
4,1029_LEV,tunedafp_ehc,"{'g': 100, 'op_list': ['n', 'v', '+', '-', '*'...",23654,610.779592,0.395927,0.486100,0.566755,0.408153,0.486601,0.545640,"{'g': 100, 'op_list': ['n', 'v', '+', '-', '*'..."
...,...,...,...,...,...,...,...,...,...,...,...,...
961,695_chatfield_4,tunedtpsr,{},23654,4498.717020,2845.082001,42.765541,-0.428028,3204.309914,43.869403,-0.517690,{}
962,695_chatfield_4,tunedtpsr,{},860,3585.596637,3075.898075,44.832459,-0.501792,4121.628553,51.347906,-1.163111,{}
963,695_chatfield_4,tunedxgboost,"{'gamma': 0.4, 'learning_rate': 0.01, 'n_estim...",15795,24.987042,100.250142,7.737974,0.950416,456.648379,14.936133,0.768606,"{'gamma': 0.4, 'learning_rate': 0.01, 'n_estim..."
964,695_chatfield_4,tunedxgboost,"{'gamma': 0.3, 'learning_rate': 0.01, 'n_estim...",23654,26.057889,84.473229,7.158334,0.957600,336.943031,12.957681,0.840410,"{'gamma': 0.3, 'learning_rate': 0.01, 'n_estim..."


In [8]:
result = df_results.groupby('algorithm')['params_str'].apply(
    lambda x: x.mode().iloc[0]).reset_index().sort_values(by='algorithm')

result.to_csv(f"{figdir}/tuned_hyperparameters.csv", index=False)
result

,algorithm,params_str
0,tunedafp,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*..."
1,tunedafp_ehc,"{'g': 1000, 'op_list': ['n', 'v', '+', '-', '*..."
2,tunedafp_fe,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*..."
3,tunedbsr,"{'max_time': 3600, 'itrNum': 500, 'treeNum': 6..."
4,tunede2et,"{'max_input_points': 50, 'max_number_bags': 1,..."
5,tunedeplex,"{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*..."
6,tunedeql,{'functions': 'id;mul;cos;sin;exp;square;sqrt;...
7,tunedfeat,"{'max_time': 3600, 'gens': 2500, 'lr': 0.1, 'p..."
8,tunedffx,{}
9,tunedgeneticengine,{'max_time': 3600}


In [9]:
# # saved tuned models.
# Go thru PMLB results and pick the most common params in the best estimators. 
# Use these parameter settings for the Feynman and Strogatz models.

# # write tuned model scripts for each algorithm

for index, row in result.iterrows():
    print(row['algorithm'])
    print(row['params_str'])
    filename = f"{params_dir}{row['algorithm'].replace('tuned','')}/_params.py"
    print(filename)
    if overwrite:
        if os.path.isfile(filename):
            os.rename(filename, f'{filename}.bak')
        with open(filename,'w') as f:
            f.write(f"# params selected based on results from: {rdir}\n")
            f.write('params = {}\n'.format(row['params_str']))
        

tunedafp
{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*', '/', 'exp', 'log', '2', '3', 'sqrt'], 'popsize': 100}
./methods/afp/_params.py
tunedafp_ehc
{'g': 1000, 'op_list': ['n', 'v', '+', '-', '*', '/', 'exp', 'log', '2', '3', 'sqrt', 'sin', 'cos'], 'popsize': 100}
./methods/afp_ehc/_params.py
tunedafp_fe
{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*', '/', 'exp', 'log', '2', '3', 'sqrt'], 'popsize': 100}
./methods/afp_fe/_params.py
tunedbsr
{'max_time': 3600, 'itrNum': 500, 'treeNum': 6, 'val': 1000}
./methods/bsr/_params.py
tunede2et
{'max_input_points': 50, 'max_number_bags': 1, 'n_trees_to_refine': 50}
./methods/e2et/_params.py
tunedeplex
{'g': 2500, 'op_list': ['n', 'v', '+', '-', '*', '/', 'exp', 'log', '2', '3', 'sqrt'], 'popsize': 100}
./methods/eplex/_params.py
tunedeql
{'functions': 'id;mul;cos;sin;exp;square;sqrt;id;mul;cos;sin;exp;square;sqrt;log', 'n_layers': 2, 'reg': 0.001}
./methods/eql/_params.py
tunedfeat
{'max_time': 3600, 'gens': 2500, 'lr': 0.1, 'pop_size': 100}
